In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.2 MB/s eta 0:00:00


In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Tokenizer loaded successfully")

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Tokenizer loaded successfully


In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded successfully")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully


In [ ]:
from datasets import Dataset

data = {
    "instruction": [
        "What is Python?",
        "What is Machine Learning?",
        "What is RAG?",
        "What is an LLM?",
        "What is fine-tuning?",
        "What is SQL?",
        "What is a neural network?",
        "What is a tokenizer?"
    ],

    "response": [
        "Python is a high-level programming language widely used for software development, data analysis, machine learning, and automation.",
        "Machine Learning is a branch of AI where computers learn patterns from data and use those patterns to make predictions or decisions.",
        "RAG stands for Retrieval-Augmented Generation. It retrieves relevant information from an external knowledge source and provides it to an LLM to generate an answer.",
        "An LLM is a Large Language Model trained on large amounts of text to understand and generate human-like language.",
        "Fine-tuning is the process of training a pretrained model further on a specific dataset so that it becomes better at a particular task or behavior.",
        "SQL is a language used to communicate with and manage relational databases.",
        "A neural network is a machine learning model made of interconnected layers that learn patterns from data.",
        "A tokenizer converts text into tokens and then into token IDs that can be processed by an LLM."
    ]
}

dataset = Dataset.from_dict(data)

print(dataset)

Dataset({
    features: ['instruction', 'response'],
    num_rows: 8
})


In [ ]:
def format_example(example):
    return {
        "text": f"""<|user|>
{example["instruction"]}
<|assistant|>
{example["response"]}"""
    }

formatted_dataset = dataset.map(format_example)

print(formatted_dataset[0])

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

{'instruction': 'What is Python?', 'response': 'Python is a high-level programming language widely used for software development, data analysis, machine learning, and automation.', 'text': '<|user|>\nWhat is Python?\n<|assistant|>\nPython is a high-level programming language widely used for software development, data analysis, machine learning, and automation.'}


In [ ]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=512
    )

    # Required for calculating training loss
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens


tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True
)

print(tokenized_dataset.column_names)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

['instruction', 'response', 'text', 'input_ids', 'attention_mask', 'labels']


In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

print("LoRA configuration created")

LoRA configuration created


In [ ]:
from peft import get_peft_model

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./fine_tuned_model",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [ ]:
trainer.train()

Step,Training Loss
1,2.084078
2,1.789465
3,1.751825
4,1.611241
5,1.734457
6,1.291217


TrainOutput(global_step=6, training_loss=1.7103802561759949, metrics={'train_runtime': 10.1596, 'train_samples_per_second': 2.362, 'train_steps_per_second': 0.591, 'total_flos': 6662253072384.0, 'train_loss': 1.7103802561759949, 'epoch': 3.0})

In [ ]:
output_dir = "./jitendra_tinyllama_lora"

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

('./shravan_tinyllama_lora/tokenizer_config.json',
 './shravan_tinyllama_lora/chat_template.jinja',
 './shravan_tinyllama_lora/tokenizer.json')

In [ ]:
import torch

prompt = """<|user|>
What is Machine Learning?
<|assistant|>
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        do_sample=True
    )

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|user|>
What is Machine Learning?
<|assistant|>
Machine learning (ML) is a field that enables computers to learn from data without being explicitly programmed. ML algorithms try to identify patterns in data and use them to make predictions or recommendations. ML can be used in various fields, including healthcare, finance, marketing, and supply chain management. Some common uses of ML include:

1. Forecasting: ML algorithms can


In [ ]:
questions = [
    "What is RAG?",
    "What is SQL?",
    "What is a tokenizer?",
    "What is a neural network?"
]

for question in questions:
    prompt = f"""<|user|>
{question}
<|assistant|>
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.7,
            do_sample=True
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("Question:", question)
    print("Answer:", response)
    print("-" * 60)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is RAG?
Answer: <|user|>
What is RAG?
<|assistant|>
RAG is a gene-editing tool developed by the EMBL-EBI (European Molecular Biology Laboratory-European Bioinformatics Institute) and the University of Oxford.
RAG stands for RNA-induced silencing complex, and it is a type of RNA editing enzyme that can edit specific RNA molecules. RAG
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is SQL?
Answer: <|user|>
What is SQL?
<|assistant|>
SQL (Structured Query Language) is a language used by database management systems to communicate with databases. It defines the structure and syntax of database queries, along with the data types and operations supported by the database. SQL is used to create, modify, and retrieve data from databases.

In other words, SQL is a language used to query and manipulate data stored in a database. It provides a
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is a tokenizer?
Answer: <|user|>
What is a tokenizer?
<|assistant|>
A tokenizer is a tool used for dividing text into smaller parts called tokens. Each token is a collection of characters or words. In natural language processing (NLP), tokenization is an important process for converting text into a sequence of tokens, which can be processed by NLP models.

For example, a sentence in English might be "The quick brown fox jumps over the
------------------------------------------------------------
Question: What is a neural network?
Answer: <|user|>
What is a neural network?
<|assistant|>
A neural network is a computational model that mimics the way the human brain processes and stores information. It consists of a series of interconnected nodes, each of which represents a specific cognitive process or feature. Neural networks can be trained using supervised or unsupervised learning techniques, and they can be used for a range of applications, including image and speech rec

In [ ]:
def chat(user_message):

    prompt = f"""<|user|>
{user_message}
<|assistant|>
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.7
        )

    # Decode only the newly generated tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
chat('what is python')

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Python is a programming language that has become a popular choice for data analysis and machine learning. It has a simple syntax and is easy to learn, making it a great choice for beginners. Python is cross-platform, meaning it works on both Mac and Windows and is available in many different languages, which makes it ideal for a wide range of applications. Python is also known for its community and open'

In [ ]:
print(chat("What is RAG?"))

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RADIO ASSISTED GENERATION (RAG) is a technology used in the field of wireless communication to improve the efficiency of wireless communication in low-density environments. It involves the use of a transmitter, a receiver, and a repeater. The transmitter sends an RF signal, which is received by the receiver. The RF signal is then amplified
